In [ ]:
!gdown 1a-QINzvrzUGEy2QDUduwSs4KSuPH0Rgr
!gdown 1ikOLo5LDQGarRgf25kLa7BkZzXd66KM-

Downloading...
From (original): https://drive.google.com/uc?id=1a-QINzvrzUGEy2QDUduwSs4KSuPH0Rgr
From (redirected): https://drive.google.com/uc?id=1a-QINzvrzUGEy2QDUduwSs4KSuPH0Rgr&confirm=t&uuid=fa043719-8884-41ea-aadc-896607af5184
To: /content/transaction_full_2025_final.parquet
100% 787M/787M [00:12<00:00, 62.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ikOLo5LDQGarRgf25kLa7BkZzXd66KM-
To: /content/items.parquet
100% 2.90M/2.90M [00:00<00:00, 23.5MB/s]


# Step 0 : Check Schema

In [ ]:
import polars as pl

transactions_path = "/content/transaction_full_2025_final.parquet"
items_path = "/content/items.parquet"

transactions_lf = pl.scan_parquet(transactions_path)
items_lf = pl.scan_parquet(items_path)

print(transactions_lf.collect_schema())
print(items_lf.collect_schema())

Schema({'customer_id': Int32, 'item_id': String, 'price': Decimal(precision=38, scale=4), 'location': Int32, 'discount': Decimal(precision=38, scale=4), 'bill_id': Int32, 'quantity': Int32, 'event_type': String, 'updated_date': Datetime(time_unit='us', time_zone=None)})
Schema({'item_id': String, 'price': Decimal(precision=38, scale=4), 'category_l1': String, 'category_l2': String, 'category_l3': String, 'category': String, 'brand': String, 'manufacturer': String, 'description': String, 'sale_status': Int32, 'size': String})


# Step 1 : Load data + chuẩn hóa cột thời gian

In [ ]:
transactions_lf = transactions_lf.with_columns([
    pl.col("updated_date").dt.date().alias("date"),
    pl.col("updated_date").dt.month().alias("month"),
    pl.col("updated_date").dt.year().alias("year"),
])

items_lf = items_lf.with_columns([
    pl.col("price").cast(pl.Float32),
])

In [ ]:
transactions_lf = transactions_lf.with_columns([
    pl.col("price").cast(pl.Float32),
    pl.col("discount").cast(pl.Float32),
])

In [ ]:
transactions_lf.select([
    pl.len().alias("n_rows"),
    pl.col("customer_id").n_unique().alias("n_customers"),
    pl.col("item_id").n_unique().alias("n_items"),
    pl.col("bill_id").n_unique().alias("n_bills"),
    pl.col("date").min().alias("min_date"),
    pl.col("date").max().alias("max_date"),
]).collect()

n_rows,n_customers,n_items,n_bills,min_date,max_date
u32,u32,u32,u32,date,date
41470317,3020253,20393,19456172,2025-01-01,2025-12-31


In [ ]:
transactions_lf.group_by("month").agg([
    pl.len().alias("n_rows"),
    pl.col("customer_id").n_unique().alias("n_customers"),
    pl.col("item_id").n_unique().alias("n_items"),
    pl.col("bill_id").n_unique().alias("n_bills"),
    pl.col("date").n_unique().alias("n_days")
]).sort("month").collect()

month,n_rows,n_customers,n_items,n_bills,n_days
i8,u32,u32,u32,u32,u32
1,3346259,657041,12457,1431408,31
2,2958962,644763,11819,1339825,28
3,3015239,635494,11952,1402760,31
4,3132138,647647,11963,1457313,30
5,3556234,751908,12795,1688419,31
…,…,…,…,…,…
8,3767332,799500,12348,1776647,31
9,3395016,744996,12362,1650477,30
10,3573167,792462,12546,1727630,30


In [ ]:
train_hist_lf = transactions_lf.filter(pl.col("month").is_between(1, 9))
train_label_lf = transactions_lf.filter(pl.col("month") == 10)

valid_hist_lf = transactions_lf.filter(pl.col("month").is_between(1, 10))
valid_label_lf = transactions_lf.filter(pl.col("month") == 12)

final_hist_lf = transactions_lf.filter(pl.col("month").is_between(1, 11))
test_target_lf = transactions_lf.filter(pl.col("month") == 12)

# Step 3 : EDA Train Data

## 3.1 Basic statistics

In [ ]:
train_hist_lf.select([
    pl.len().alias("n_rows"),
    pl.col("customer_id").n_unique().alias("n_customers"),
    pl.col("item_id").n_unique().alias("n_items"),
    pl.col("bill_id").n_unique().alias("n_bills"),
]).collect()

n_rows,n_customers,n_items,n_bills
u32,u32,u32,u32
30067401,2433698,18601,13950467


## 3.2 Basket size analysis

In [ ]:
basket_stats = (
    train_hist_lf
    .group_by("bill_id")
    .agg([
        pl.col("item_id").n_unique().alias("basket_size"),
        pl.col("quantity").sum().alias("total_quantity"),
    ])
)

basket_stats.select([
    pl.col("basket_size").mean().alias("avg_basket_size"),
    pl.col("basket_size").median().alias("median_basket_size"),
    pl.col("basket_size").quantile(0.95).alias("p95_basket_size"),
    pl.col("basket_size").max().alias("max_basket_size"),
]).collect()

avg_basket_size,median_basket_size,p95_basket_size,max_basket_size
f64,f64,f64,u32
2.154227,1.0,6.0,84


In [ ]:
customer_stats = (
    train_hist_lf
    .group_by("customer_id")
    .agg([
        pl.len().alias("n_transactions"),
        pl.col("bill_id").n_unique().alias("n_bills"),
        pl.col("item_id").n_unique().alias("n_unique_items"),
        pl.col("quantity").sum().alias("total_quantity"),
    ])
)

customer_stats.select([
    pl.col("n_bills").mean().alias("avg_bills"),
    pl.col("n_bills").median().alias("median_bills"),
    pl.col("n_bills").quantile(0.95).alias("p95_bills"),
]).collect()

avg_bills,median_bills,p95_bills
f64,f64,f64
5.73221,2.0,25.0


In [ ]:
item_popularity = (
    train_hist_lf
    .group_by("item_id")
    .agg([
        pl.len().alias("n_transactions"),
        pl.col("customer_id").n_unique().alias("n_customers"),
        pl.col("quantity").sum().alias("total_quantity"),
    ])
    .sort("n_transactions", descending=True)
)


item_popularity.head(20).collect()

item_id,n_transactions,n_customers,total_quantity
str,u32,u32,i32
"""4690000000001""",424986,242379,1079177
"""7176000000002""",186459,71316,1326056
"""1512000000004""",165840,111732,192082
"""2803000000013""",152988,87867,170390
"""4603024000002""",137592,88503,198520
…,…,…,…
"""0020010000438""",102837,31746,132036
"""0029130000030""",102743,55735,122302
"""5444000000016""",102644,75131,109576


In [ ]:
user_item_repeat = (
    train_hist_lf
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.col("bill_id").n_unique().alias("n_bills")
    ])
)

repeat_stats = (
    user_item_repeat
    .group_by("item_id")
    .agg([
        (pl.col("n_bills") > 1).mean().alias("repeat_rate")
    ])
    .sort("repeat_rate", descending=True)
)

repeat_stats.head(20).collect()

item_id,repeat_rate
str,f64
"""3523000000043""",1.0
"""0972000000034""",1.0
"""3502000000018""",1.0
"""0009010160375""",1.0
"""0921285490007""",1.0
…,…
"""2070000000005""",1.0
"""0884004760001""",1.0
"""0007010000778""",1.0


# Step 4 Candidate Generation

## Step 4.0 Tạo active users

In [ ]:
active_train_users_lf = (
    train_hist_lf
    .group_by("customer_id")
    .agg(
        pl.col("bill_id").n_unique().alias("n_bills")
    )
    .filter(pl.col("n_bills") >= 2)
    .select("customer_id")
)
active_valid_users_lf = (
    valid_hist_lf
    .group_by("customer_id")
    .agg(
        pl.col("bill_id").n_unique().alias("n_bills")
    )
    .filter(pl.col("n_bills") >= 2)
    .select("customer_id")
)

active_final_users_lf = (
    final_hist_lf
    .group_by("customer_id")
    .agg(
        pl.col("bill_id").n_unique().alias("n_bills")
    )
    .filter(pl.col("n_bills") >= 2)
    .select("customer_id")
)

In [ ]:
active_train_users_lf.select(pl.len().alias("n_active_train_users")).collect()
active_valid_users_lf.select(pl.len().alias("n_active_valid_users")).collect()
active_final_users_lf.select(pl.len().alias("n_active_final_users")).collect()

n_active_final_users
u32
1648890


## Step 4.1 Candidate source 1: Recently Purchased Items

In [ ]:
recent_candidates_train_lf = (
    train_hist_lf
    .join(active_train_users_lf, on="customer_id", how="inner")
    .sort(["customer_id", "updated_date"], descending=[False, True])
    .group_by("customer_id")
    .agg(
        pl.col("item_id").unique().head(20).alias("candidate_items")
    )
    .explode("candidate_items")
    .rename({"candidate_items": "item_id"})
    .with_columns(
        pl.lit("recent").alias("candidate_source")
    )
)

In [ ]:
recent_candidates_train_lf.head(30).collect()

customer_id,item_id,candidate_source
i32,str,str
17212,"""3509000000098""","""recent"""
17212,"""6048000000106""","""recent"""
17212,"""3775000000003""","""recent"""
17266,"""0020020000185""","""recent"""
17266,"""2261000000005""","""recent"""
…,…,…
28879,"""0068000000160""","""recent"""
28879,"""0068000000159""","""recent"""
28879,"""7161000000001""","""recent"""


## Step 4.2 Candidate source 2: Frequently Purchased Items

In [ ]:
freq_candidates_train_lf = (
    train_hist_lf
    .join(active_train_users_lf, on="customer_id", how="inner")
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("n_transactions"),
        pl.col("quantity").sum().alias("total_quantity"),
    ])
    .sort(["customer_id", "n_transactions", "total_quantity"], descending=[False, True, True])
    .group_by("customer_id")
    .agg(
        pl.col("item_id").head(20).alias("candidate_items")
    )
    .explode("candidate_items")
    .rename({"candidate_items": "item_id"})
    .with_columns(
        pl.lit("frequent").alias("candidate_source")
    )
)

In [ ]:
freq_candidates_train_lf.head(30).collect()

customer_id,item_id,candidate_source
i32,str,str
17212,"""3775000000003""","""frequent"""
17212,"""6048000000106""","""frequent"""
17212,"""3509000000098""","""frequent"""
17266,"""2261000000005""","""frequent"""
17266,"""4280000000031""","""frequent"""
…,…,…
28879,"""6849000000002""","""frequent"""
28879,"""2610000000004""","""frequent"""
28879,"""3052000000001""","""frequent"""


## Step 4.3 Candidate source 3: Popular Items

In [ ]:
top_popular_items_lf = (
    train_hist_lf
    .group_by("item_id")
    .agg([
        pl.len().alias("n_transactions"),
        pl.col("customer_id").n_unique().alias("n_customers"),
        pl.col("quantity").sum().alias("total_quantity"),
    ])
    .sort(["n_customers", "n_transactions"], descending=[True, True])
    .head(50)
    .select("item_id")
)

popular_candidates_train_lf = (
    active_train_users_lf
    .join(top_popular_items_lf, how="cross")
    .with_columns(
        pl.lit("popular").alias("candidate_source")
    )
)

In [ ]:
popular_candidates_train_lf.head(30).collect()

customer_id,item_id,candidate_source
i32,str,str
9004197,"""4690000000001""","""popular"""
9004197,"""1512000000004""","""popular"""
9004197,"""4603024000001""","""popular"""
9004197,"""5950000000001""","""popular"""
9004197,"""4603024000002""","""popular"""
…,…,…
9004197,"""0029130000029""","""popular"""
9004197,"""3880000000002""","""popular"""
9004197,"""2707000000001""","""popular"""


## Step 4.4 Gộp candidate baseline

In [ ]:
train_candidates_lf = (
    pl.concat([
        recent_candidates_train_lf,
        freq_candidates_train_lf,
    ])
    .unique(["customer_id", "item_id"])
)

train_candidates_lf.sink_parquet("/content/train_candidates.parquet")

In [ ]:
train_candidates_lf.select([
    pl.len().alias("n_candidates"),
    pl.col("customer_id").n_unique().alias("n_users"),
    pl.col("item_id").n_unique().alias("n_items"),
]).collect()

n_candidates,n_users,n_items
u32,u32,u32
15433031,1427686,17468


# Step 5 : Tạo label tháng 10

## Step 5.1 Tạo ground truth tháng 10

In [ ]:
train_gt_lf = (
    train_label_lf
    .select(["customer_id", "item_id"])
    .unique()
    .with_columns(
        pl.lit(1, dtype=pl.Int8).alias("target")
    )
)

## Step 5.2 Join candidate với ground truth

In [ ]:
train_dataset_lf = (
    train_candidates_lf
    .select([
        pl.col("customer_id").cast(pl.Int32),
        pl.col("item_id").cast(pl.String),
    ])
    .join(
        train_gt_lf.select([
            pl.col("customer_id").cast(pl.Int32),
            pl.col("item_id").cast(pl.String),
            pl.col("target").cast(pl.Int8),
        ]),
        on=["customer_id", "item_id"],
        how="left"
    )
    .with_columns(
        pl.col("target")
        .fill_null(pl.lit(0, dtype=pl.Int8))
        .cast(pl.Int8)
    )
)

## Step 5.3 Ghi ra parquet

In [ ]:
train_dataset_lf.sink_parquet("/content/train_dataset_labels.parquet")

In [ ]:
train_dataset_lf = pl.scan_parquet("/content/train_dataset_labels.parquet")

In [ ]:
train_dataset_lf.group_by("target").agg(
    pl.len().alias("count")
).collect()

target,count
i8,u32
0,14657069
1,775691


# Step 6 — Feature Engineering

## Step 6.1 User features

In [ ]:
user_features_lf = (
    train_hist_lf
    .group_by("customer_id")
    .agg([
        pl.len().alias("user_n_transactions"),
        pl.col("bill_id").n_unique().alias("user_n_bills"),
        pl.col("item_id").n_unique().alias("user_n_unique_items"),
        pl.col("quantity").sum().alias("user_total_quantity"),
        pl.col("date").n_unique().alias("user_n_active_days"),
        pl.col("price").mean().alias("user_avg_price"),
        pl.col("discount").mean().alias("user_avg_discount"),
    ])
    .with_columns([
        (
            pl.col("user_n_transactions") / pl.col("user_n_bills")
        ).alias("user_avg_items_per_bill"),

        (
            pl.col("user_total_quantity") / pl.col("user_n_bills")
        ).alias("user_avg_quantity_per_bill"),
    ])
)

## Step 6.2 Item features

In [ ]:
item_features_lf = (
    train_hist_lf
    .group_by("item_id")
    .agg([
        pl.len().alias("item_n_transactions"),
        pl.col("customer_id").n_unique().alias("item_n_customers"),
        pl.col("bill_id").n_unique().alias("item_n_bills"),
        pl.col("quantity").sum().alias("item_total_quantity"),
        pl.col("price").mean().alias("item_avg_price"),
        pl.col("discount").mean().alias("item_avg_discount"),
    ])
)

In [ ]:
item_meta_lf = (
    items_lf
    .select([
        "item_id",
        "category_l1",
        "category_l2",
        "category_l3",
        "category",
        "brand",
        "manufacturer",
        "sale_status",
        "size",
    ])
)

In [ ]:
item_features_lf = (
    item_features_lf
    .join(item_meta_lf, on="item_id", how="left")
)

## Step 6.3 User-item interaction features

In [ ]:
user_item_features_lf = (
    train_hist_lf
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("ui_n_transactions"),
        pl.col("bill_id").n_unique().alias("ui_n_bills"),
        pl.col("quantity").sum().alias("ui_total_quantity"),
        pl.col("date").max().alias("ui_last_date"),
        pl.col("date").min().alias("ui_first_date"),
    ])
    .with_columns([
        (
            pl.lit(train_hist_lf.select(pl.col("date").max()).collect()[0, 0])
            - pl.col("ui_last_date")
        ).dt.total_days().alias("ui_recency_days")
    ])
)

In [ ]:
user_item_features_lf.head(5).collect()

customer_id,item_id,ui_n_transactions,ui_n_bills,ui_total_quantity,ui_last_date,ui_first_date,ui_recency_days
i32,str,u32,u32,i32,date,date,i64
7879969,"""0007090000252""",1,1,1,2025-01-10,2025-01-10,263
8450160,"""7176000000002""",1,1,1,2025-03-23,2025-03-23,191
8605859,"""5048000000005""",1,1,1,2025-07-30,2025-07-30,62
6774956,"""5506000000004""",1,1,1,2025-01-20,2025-01-20,253
602679,"""7167000000005""",2,2,2,2025-08-25,2025-07-28,36


In [ ]:
user_features_lf.head(5).collect()

customer_id,user_n_transactions,user_n_bills,user_n_unique_items,user_total_quantity,user_n_active_days,user_avg_price,user_avg_discount,user_avg_items_per_bill,user_avg_quantity_per_bill
i32,u32,u32,u32,i32,u32,f32,f32,f64,f64
1319537,1,1,1,1,1,69000.0,0.0,1.0,1.0
8694761,1,1,1,3,1,285000.0,0.0,1.0,3.0
8985234,1,1,1,1,1,255000.0,0.0,1.0,1.0
554791,15,7,13,17,6,88456.664062,2330.0,2.142857,2.428571
6854804,53,18,25,145,16,97668.492188,3066.415039,2.944444,8.055556


In [ ]:
item_features_lf.head(5).collect()

item_id,item_n_transactions,item_n_customers,item_n_bills,item_total_quantity,item_avg_price,item_avg_discount,category_l1,category_l2,category_l3,category,brand,manufacturer,sale_status,size
str,u32,u32,u32,i32,f32,f32,str,str,str,str,str,str,i32,str
"""3953000000565""",616,598,616,626,93650.046875,35905.046875,"""Thời trang""","""Thời trang bé trai""","""Bộ bé trai""","""Bộ bé trai Animo Easy""","""Animo""","""Không xác định""",1,"""Không xác định"""
"""3944000000395""",80,79,80,80,157378.71875,71746.289062,"""Thời trang""","""Thời trang bé trai""","""Bộ bé trai""","""Bộ bé trai Animo""","""Animo""","""Không xác định""",0,"""Không xác định"""
"""4351000000002""",49,49,49,50,35419.875,49.512527,"""Hóa mỹ phẩm cho bé""","""Chăm sóc sức khỏe bé""","""Tai mũi họng""","""Xịt mũi Bisalt & Hymasalt""","""BISALT""","""Không xác định""",0,"""Không xác định"""
"""3953000000081""",3,3,3,3,112500.0,66500.0,"""Thời trang""","""Thời trang bé gái""","""Đầm bé gái""","""Đầm bé gái Animo""","""Animo""","""Không xác định""",0,"""Không xác định"""
"""7446000000001""",18,17,18,19,195864.046875,271.906769,"""Sữa""","""Snow Brand""","""Snow Brand""","""Snow Brand Step 1""","""Megmilk Snow Brand""","""Không xác định""",1,"""Không xác định"""


## Step 6.4 Join feature vào train dataset

In [ ]:
user_features_lf.sink_parquet("/content/user_features.parquet")
item_features_lf.sink_parquet("/content/item_features.parquet")
user_item_features_lf.sink_parquet("/content/user_item_features.parquet")

In [ ]:
user_features_lf = pl.scan_parquet("/content/user_features.parquet")
item_features_lf = pl.scan_parquet("/content/item_features.parquet")
user_item_features_lf = pl.scan_parquet("/content/user_item_features.parquet")

In [ ]:
train_features_step1_lf = (
    train_dataset_lf
    .join(user_features_lf, on="customer_id", how="left")
)

train_features_step1_lf.sink_parquet("/content/train_features_step1.parquet")

In [ ]:
train_features_step1_lf = pl.scan_parquet("/content/train_features_step1.parquet")

In [ ]:
train_features_step2_lf = (
    train_features_step1_lf
    .join(item_features_lf, on="item_id", how="left")
)

train_features_step2_lf.sink_parquet("/content/train_features_step2.parquet")

In [ ]:
train_features_step2_lf = pl.scan_parquet("/content/train_features_step2.parquet")

In [ ]:
train_features_lf = (
    train_features_step2_lf
    .join(user_item_features_lf, on=["customer_id", "item_id"], how="left")
    .with_columns([
        pl.col("ui_n_transactions").fill_null(0).cast(pl.Int32),
        pl.col("ui_n_bills").fill_null(0).cast(pl.Int32),
        pl.col("ui_total_quantity").fill_null(0).cast(pl.Int32),
        pl.col("ui_recency_days").fill_null(9999).cast(pl.Int32),
    ])
)

train_features_lf.sink_parquet("/content/train_features.parquet")

In [ ]:
!gdown 1MzYhbF_PBAE1H99ujlaX8fi8iLJkY3ic

Downloading...
From (original): https://drive.google.com/uc?id=1MzYhbF_PBAE1H99ujlaX8fi8iLJkY3ic
From (redirected): https://drive.google.com/uc?id=1MzYhbF_PBAE1H99ujlaX8fi8iLJkY3ic&confirm=t&uuid=d81158a7-4e71-45e3-ac6b-c112691631ba
To: /content/train_features.parquet
100% 643M/643M [00:08<00:00, 73.8MB/s]


In [ ]:
train_features_lf = pl.scan_parquet("/content/train_features.parquet")
train_features_lf.head(5).collect().write_csv("sample_train_feature.csv")

# Step 7 Prepare train matrix

## 7.1 Chọn feature columns

In [ ]:
drop_cols = [
    "customer_id",
    "item_id",
    "target",
    "ui_last_date",
    "ui_first_date",
    "description",
]

cat_cols = [
    "category_l1",
    "category_l2",
    "category_l3",
    "category",
    "brand",
    "manufacturer",
    "size",
]

numeric_cols = [
    c for c, dtype in train_features_lf.collect_schema().items()
    if c not in drop_cols + cat_cols
]

## Step 7.2 Encode categorical bằng Polars

In [ ]:
train_model_lf = (
    train_features_lf
    .with_columns([
        pl.col(c).fill_null("unknown").cast(pl.Categorical).to_physical().alias(c)
        for c in cat_cols
    ])
    .with_columns([
        pl.col(c).fill_null(0)
        for c in numeric_cols
    ])
    .select(
        ["target"] + numeric_cols + cat_cols
    )
)

In [ ]:
train_model_lf.sink_parquet("/content/train_model_ready.parquet")

In [ ]:
print(train_model_lf.head(5).collect())

shape: (5, 28)
┌────────┬──────────────┬──────────────┬──────────────┬───┬──────────┬───────┬──────────────┬──────┐
│ target ┆ user_n_trans ┆ user_n_bills ┆ user_n_uniqu ┆ … ┆ category ┆ brand ┆ manufacturer ┆ size │
│ ---    ┆ actions      ┆ ---          ┆ e_items      ┆   ┆ ---      ┆ ---   ┆ ---          ┆ ---  │
│ i8     ┆ ---          ┆ u32          ┆ ---          ┆   ┆ u32      ┆ u32   ┆ u32          ┆ u32  │
│        ┆ u32          ┆              ┆ u32          ┆   ┆          ┆       ┆              ┆      │
╞════════╪══════════════╪══════════════╪══════════════╪═══╪══════════╪═══════╪══════════════╪══════╡
│ 0      ┆ 21           ┆ 14           ┆ 19           ┆ … ┆ 464      ┆ 715   ┆ 1542         ┆ 1542 │
│ 0      ┆ 39           ┆ 22           ┆ 32           ┆ … ┆ 465      ┆ 1532  ┆ 1542         ┆ 1542 │
│ 1      ┆ 36           ┆ 24           ┆ 17           ┆ … ┆ 466      ┆ 16    ┆ 1542         ┆ 1542 │
│ 0      ┆ 29           ┆ 9            ┆ 23           ┆ … ┆ 467      ┆ 1533 

In [ ]:
import pandas as pd

train_df = pd.read_parquet("/content/train_model_ready.parquet")

In [ ]:
train_df.shape, train_df["target"].value_counts()

((15432760, 28),
 target
 0    14657069
 1      775691
 Name: count, dtype: int64)

In [ ]:
X = train_df.drop(columns=["target"])
cl = X.columns

In [ ]:
del X
del train_df

# Step 8 Train LightGBM baseline

In [ ]:
!pip install lightgbm -q

## 8.1 Tách X/y

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = train_df["target"]
X = train_df.drop(columns=["target"])

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

## 8.2 Train model binary baseline

In [ ]:
pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos

model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_holdout, y_holdout)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50),
    ],
)

Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.883233	valid_0's binary_logloss: 0.386772
Early stopping, best iteration is:
[2]	valid_0's auc: 0.871255	valid_0's binary_logloss: 0.181791


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
               n_jobs=-1, num_leaves=63, objective='binary', random_state=42,
               scale_pos_weight=np.float64(18.895496775635205), subsample=0.8,
               verbose=-1)

## 8.3 Check AUC tạm thời

In [ ]:
pred_holdout = model.predict_proba(X_holdout)[:, 1]

auc = roc_auc_score(y_holdout, pred_holdout)
print("Holdout AUC:", auc)

Holdout AUC: 0.8712553608502811


## 8.4 Feature importance

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df.head(30)

,feature,importance
20,category_l1,20
19,ui_recency_days,19
9,item_n_transactions,14
13,item_avg_price,10
4,user_n_active_days,9
10,item_n_customers,8
15,sale_status,7
22,category_l3,6
3,user_total_quantity,5
14,item_avg_discount,5


In [ ]:
import joblib

joblib.dump(model, "/content/lgbm_baseline.pkl")

['/content/lgbm_baseline.pkl']

# Step 9 — Validation setup

## 9.1 Active users validation

In [ ]:
active_valid_users_lf = (
    valid_hist_lf
    .group_by("customer_id")
    .agg(pl.col("bill_id").n_unique().alias("n_bills"))
    .filter(pl.col("n_bills") >= 2)
    .select("customer_id")
)

## 9.2 Recent candidates validation

In [ ]:
recent_candidates_valid_lf = (
    valid_hist_lf
    .join(active_valid_users_lf, on="customer_id", how="inner")
    .sort(["customer_id", "updated_date"], descending=[False, True])
    .group_by("customer_id")
    .agg(
        pl.col("item_id").head(30).alias("candidate_items")
    )
    .explode("candidate_items")
    .rename({"candidate_items": "item_id"})
    .unique(["customer_id", "item_id"])
)

## 9.3 Frequent candidates validation

In [ ]:
freq_candidates_valid_lf = (
    valid_hist_lf
    .join(active_valid_users_lf, on="customer_id", how="inner")
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("n_transactions"),
        pl.col("quantity").sum().alias("total_quantity"),
    ])
    .sort(["customer_id", "n_transactions", "total_quantity"], descending=[False, True, True])
    .group_by("customer_id")
    .agg(
        pl.col("item_id").head(20).alias("candidate_items")
    )
    .explode("candidate_items")
    .rename({"candidate_items": "item_id"})
    .unique(["customer_id", "item_id"])
)

## 9.4 Merge validation candidates

In [ ]:
recent_candidates_valid_lf.sink_parquet("/content/recent_candidates_valid.parquet")
freq_candidates_valid_lf.sink_parquet("/content/freq_candidates_valid.parquet")

In [ ]:
valid_candidates_lf = (
    pl.concat([
        pl.scan_parquet("/content/recent_candidates_valid.parquet"),
        pl.scan_parquet("/content/freq_candidates_valid.parquet"),
    ])
    .unique(["customer_id", "item_id"])
)

valid_candidates_lf.sink_parquet("/content/valid_candidates.parquet")
valid_candidates_lf = pl.scan_parquet("/content/valid_candidates.parquet")

## 9.5 Validation label tháng 11

In [ ]:
valid_gt_lf = (
    valid_label_lf
    .select(["customer_id", "item_id"])
    .unique()
    .with_columns(
        pl.lit(1, dtype=pl.Int8).alias("target")
    )
)

valid_gt_lf.sink_parquet('/content/valid_gt.parquet')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil

src = "/content/valid_gt.parquet"
dst = "/content/drive/MyDrive/PyThongML/valid_gt.parquet"

shutil.copy(src, dst)

'/content/drive/MyDrive/PyThongML/valid_gt.parquet'

In [ ]:
valid_dataset_lf = (
    valid_candidates_lf
    .select([
        pl.col("customer_id").cast(pl.Int32),
        pl.col("item_id").cast(pl.String),
    ])
    .join(
        valid_gt_lf.select([
            pl.col("customer_id").cast(pl.Int32),
            pl.col("item_id").cast(pl.String),
            pl.col("target").cast(pl.Int8),
        ]),
        on=["customer_id", "item_id"],
        how="left"
    )
    .with_columns(
        pl.col("target").fill_null(pl.lit(0, dtype=pl.Int8)).cast(pl.Int8)
    )
)

valid_dataset_lf.sink_parquet("/content/valid_dataset_labels.parquet")
valid_dataset_lf = pl.scan_parquet("/content/valid_dataset_labels.parquet")

In [ ]:
valid_dataset_lf.group_by("target").agg(
    pl.len().alias("count")
).collect()

target,count
i8,u32
1,828925
0,17876729


## Step 9.6 — Tạo validation features

### 1. User features

In [ ]:
valid_user_features_lf = (
    valid_hist_lf
    .group_by("customer_id")
    .agg([
        pl.len().alias("user_n_transactions"),
        pl.col("bill_id").n_unique().alias("user_n_bills"),
        pl.col("item_id").n_unique().alias("user_n_unique_items"),
        pl.col("quantity").sum().alias("user_total_quantity"),
        pl.col("date").n_unique().alias("user_n_active_days"),
        pl.col("price").mean().alias("user_avg_price"),
        pl.col("discount").mean().alias("user_avg_discount"),
    ])
    .with_columns([
        (pl.col("user_n_transactions") / pl.col("user_n_bills")).alias("user_avg_items_per_bill"),
        (pl.col("user_total_quantity") / pl.col("user_n_bills")).alias("user_avg_quantity_per_bill"),
    ])
)

### 2. Item features

In [ ]:
valid_item_features_lf = (
    valid_hist_lf
    .group_by("item_id")
    .agg([
        pl.len().alias("item_n_transactions"),
        pl.col("customer_id").n_unique().alias("item_n_customers"),
        pl.col("bill_id").n_unique().alias("item_n_bills"),
        pl.col("quantity").sum().alias("item_total_quantity"),
        pl.col("price").mean().alias("item_avg_price"),
        pl.col("discount").mean().alias("item_avg_discount"),
    ])
    .join(item_meta_lf, on="item_id", how="left")
)

### 3. User-item features

In [ ]:
valid_max_date = valid_hist_lf.select(
    pl.col("date").max()
).collect()[0, 0]

valid_user_item_features_lf = (
    valid_hist_lf
    .group_by(["customer_id", "item_id"])
    .agg([
        pl.len().alias("ui_n_transactions"),
        pl.col("bill_id").n_unique().alias("ui_n_bills"),
        pl.col("quantity").sum().alias("ui_total_quantity"),
        pl.col("date").max().alias("ui_last_date"),
        pl.col("date").min().alias("ui_first_date"),
    ])
    .with_columns([
        (pl.lit(valid_max_date) - pl.col("ui_last_date")).dt.total_days().alias("ui_recency_days")
    ])
)

In [ ]:
valid_user_features_lf.sink_parquet("/content/valid_user_features.parquet")
valid_item_features_lf.sink_parquet("/content/valid_item_features.parquet")
valid_user_item_features_lf.sink_parquet("/content/valid_user_item_features.parquet")

valid_user_features_lf = pl.scan_parquet("/content/valid_user_features.parquet")
valid_item_features_lf = pl.scan_parquet("/content/valid_item_features.parquet")
valid_user_item_features_lf = pl.scan_parquet("/content/valid_user_item_features.parquet")

## 5. Join từng bước

In [ ]:
valid_features_step1_lf = (
    valid_dataset_lf
    .join(valid_user_features_lf, on="customer_id", how="left")
)

valid_features_step1_lf.sink_parquet("/content/valid_features_step1.parquet")
valid_features_step1_lf = pl.scan_parquet("/content/valid_features_step1.parquet")

In [ ]:
valid_features_step2_lf = (
    valid_features_step1_lf
    .join(valid_item_features_lf, on="item_id", how="left")
)

valid_features_step2_lf.sink_parquet("/content/valid_features_step2.parquet")
valid_features_step2_lf = pl.scan_parquet("/content/valid_features_step2.parquet")

In [ ]:
valid_features_lf = (
    valid_features_step2_lf
    .join(valid_user_item_features_lf, on=["customer_id", "item_id"], how="left")
    .with_columns([
        pl.col("ui_n_transactions").fill_null(0).cast(pl.Int32),
        pl.col("ui_n_bills").fill_null(0).cast(pl.Int32),
        pl.col("ui_total_quantity").fill_null(0).cast(pl.Int32),
        pl.col("ui_recency_days").fill_null(9999).cast(pl.Int32),
    ])
)

valid_features_lf.sink_parquet("/content/valid_features.parquet")
valid_features_lf = pl.scan_parquet("/content/valid_features.parquet")

In [ ]:
valid_features_lf.head(5).collect()

customer_id,item_id,target,user_n_transactions,user_n_bills,user_n_unique_items,user_total_quantity,user_n_active_days,user_avg_price,user_avg_discount,user_avg_items_per_bill,user_avg_quantity_per_bill,item_n_transactions,item_n_customers,item_n_bills,item_total_quantity,item_avg_price,item_avg_discount,category_l1,category_l2,category_l3,category,brand,manufacturer,sale_status,size,ui_n_transactions,ui_n_bills,ui_total_quantity,ui_last_date,ui_first_date,ui_recency_days
i32,str,i8,u32,u32,u32,i32,u32,f32,f32,f64,f64,u32,u32,u32,i32,f32,f32,str,str,str,str,str,str,i32,str,i32,i32,i32,date,date,i32
3575986,"""7451000000002""",0,24,12,16,29,12,124818.75,2527.083252,2.0,2.416667,12479,9420,12476,26467,12293.699219,5235.222168,"""Thực phẩm cho bé""","""TP từ sữa (bảo quản thường)""","""Sữa chua uống""","""Morinaga""","""Morinaga""","""Công ty cổ phần Morinaga Nutri…",1,"""Không xác định""",1,1,2,2025-08-01,2025-08-01,121
3575986,"""2700000000002""",0,24,12,16,29,12,124818.75,2527.083252,2.0,2.416667,121360,69599,120801,202989,57661.148438,13946.193359,"""Vệ sinh""","""Khăn khô""","""Khăn vải khô""","""Khăn vải khô Animo""","""Animo""","""Hebei Jiating Health Product C…",1,"""Không xác định""",1,1,2,2025-01-24,2025-01-24,310
3576170,"""1455000000004""",0,53,14,37,57,13,70015.28125,4097.924316,3.785714,4.071429,23535,16893,23533,25786,61489.550781,2202.296143,"""Thực phẩm cho bé""","""Mì & Đồ khô ăn liền""","""Rắc""","""Ruốc""","""US FOOD""","""Chi nhánh công ty TNHH Quốc Tế…",1,"""Không xác định""",1,1,1,2025-10-28,2025-10-28,33
3576170,"""0029250010001""",0,53,14,37,57,13,70015.28125,4097.924316,3.785714,4.071429,82507,46920,82489,91810,74036.398438,5529.86084,"""Thực phẩm cho bé""","""Snack ăn dặm""","""Sữa chua khô""","""Ivenet""","""Ivenet""","""Không xác định""",1,"""Không xác định""",2,2,2,2025-10-15,2025-08-15,46
3576483,"""0203000000002""",0,10,8,8,11,8,90800.0,3000.0,1.25,1.375,34210,27943,34188,41163,34396.035156,689.448486,"""Babycare""","""Đồ dùng vệ sinh""","""Vệ sinh cơ thể""","""Tăm bông, băng rốn""","""Jomi""","""Không xác định""",0,"""Không xác định""",1,1,1,2025-04-08,2025-04-08,236


## Step 9.7 — Predict validation + Precision@10

### 1. Encode validation giống train

In [ ]:
cat_cols = [
    "category_l1",
    "category_l2",
    "category_l3",
    "category",
    "brand",
    "manufacturer",
    "size",
]

numeric_cols = [
    c for c, dtype in train_features_lf.collect_schema().items()
    if c not in drop_cols + cat_cols
]

valid_model_lf = (
    valid_features_lf
    .with_columns([
        pl.col(c).fill_null("unknown").cast(pl.Categorical).to_physical().alias(c)
        for c in cat_cols
    ])
    .with_columns([
        pl.col(c).fill_null(0)
        for c in numeric_cols
    ])
    .select(
        ["customer_id", "item_id", "target"] + numeric_cols + cat_cols
    )
)

valid_model_lf.sink_parquet("/content/valid_model_ready.parquet")

### 2. Đọc bằng pandas

In [ ]:
!gdown 1xUBAe738-Fym4dacmpY5JBjXnvPbmwu2

Downloading...
From (original): https://drive.google.com/uc?id=1xUBAe738-Fym4dacmpY5JBjXnvPbmwu2
From (redirected): https://drive.google.com/uc?id=1xUBAe738-Fym4dacmpY5JBjXnvPbmwu2&confirm=t&uuid=47ce8e6a-9d63-498a-9e64-78aee04eda5a
To: /content/valid_model_ready.parquet
100% 683M/683M [00:10<00:00, 63.2MB/s]


In [ ]:
import pandas as pd
valid_df = pd.read_parquet("/content/valid_model_ready.parquet")

id_cols = valid_df[["customer_id", "item_id", "target"]]
X_valid = valid_df.drop(columns=["customer_id", "item_id", "target"])

In [ ]:
X_valid = X_valid[cl]

In [ ]:
!gdown 10Pnpq65NAv3TdovO1GfxYWXOTD3lTty_

Downloading...
From: https://drive.google.com/uc?id=10Pnpq65NAv3TdovO1GfxYWXOTD3lTty_
To: /content/lgbm_baseline.pkl
100% 21.9k/21.9k [00:00<00:00, 31.6MB/s]


In [ ]:
import joblib

model = joblib.load("/content/lgbm_baseline.pkl")

In [ ]:
valid_df["score"] = model.predict_proba(X_valid)[:, 1]

In [ ]:
valid_top10 = (
    valid_df
    .sort_values(["customer_id", "score"], ascending=[True, False])
    .groupby("customer_id")
    .head(10)
)

In [ ]:
precision_at_10 = (
    valid_top10
    .groupby("customer_id")["target"]
    .sum()
    .mean() / 10
)

print("Precision@10:", precision_at_10)

Precision@10: 0.04323527536763606


--- Test tính precision khác ----

In [ ]:
valid_buyers = set(
    valid_label_lf
    .select("customer_id")
    .unique()
    .collect()["customer_id"]
)

In [ ]:
valid_top10_eval = valid_top10[
    valid_top10["customer_id"].isin(valid_buyers)
]

In [ ]:
precision_at_10 = (
    valid_top10_eval
    .groupby("customer_id")["target"]
    .sum()
    .mean() / 10
)

In [ ]:
print("Precision@10:", precision_at_10)

Precision@10: 0.0998794174263611
